# N154 · SQL子查询、CTE与反连接

**目标：** 把复杂查询拆成可验证中间关系。

**先修：** N153。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** EXISTS/NOT EXISTS；相关子查询；CTE；UNION/UNION ALL；去重；关系除法扩展。

**配套讲解来源：** [同名逐章意见](../../comment/154_sql_subqueries_cte_antijoins.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

查询一复杂，一口气写完就没人看得懂、也没法验证。本章的主题是**把大问题拆成有名有姓、可以单独验证的小块**：子查询（查询里套查询）、CTE（给中间结果起名字）、反连接（找"配不上记录"的行）。具体三道题：

1. **从不订购的客户**：找出从来没有下过订单的客户名。用 “分段实现”部分 的数据：Customers 有 A(1)、B(2)、C(3)，Orders 有 `(id1,客户1)`、`(id2,客户NULL)`、`(id3,客户1)` 三行。输入 → 输出 `{B, C}` → 为什么：A 下过两单（订单 1 和 3 都指向他），B 和 C 一单都没有。注意中间那行"客户为 NULL"的订单是故意埋的雷——它不属于任何人，但能把某些写法炸得颗粒无收（见下文 NOT IN 陷阱）。
2. **部门最高工资**：每个部门工资最高的人（们）。R&D 部门有 Ada(100)、Ben(100)、Cal(90)。输入 → 输出 `{'R&D' 的 Ada(100) 和 Ben(100)，'Sales' 的 Dee(80)}` → 为什么：R&D 的最高薪是 100，**两个人并列**，都得保留，不能随机留一个。
3. **买齐所有产品的客户**：Product 表有产品 1、2；购买记录里客户 1 买了 `[1,1,2]`、客户 2 买了 `[1,1]`、客户 3 买了 `[2]`。输入 → 输出 `{1}` → 为什么：只有客户 1 把两种产品都买过（重复买不要紧，看的是"有没有买过"）；客户 2 缺产品 2，客户 3 缺产品 1。

这三题的共同味道：都在问"存在/不存在某类关联记录"，而 SQL 里表达"不存在"恰恰坑最多。先把三个查询的骨架摆在一起看：

| 查询 | 结构 | 关键武器 | 专治的坑 |
|---|---|---|---|
| 从不订购 | 相关子查询 | `NOT EXISTS` | NOT IN 遇 NULL 全灭 |
| 部门最高薪 | CTE + 双键连接 | `WITH ... AS` + `MAX` | 并列最高薪被丢弃 |
| 买齐所有产品 | 双重 NOT EXISTS | 关系除法 + `DISTINCT` | "所有"语义没法用简单连接表达 |

三题难度递进：第一题一层否定，第二题拆中间表，第三题两层否定嵌套——正好是从"会用子查询"到"会拆复杂语义"的路径。

## 2. 基础知识：先读懂这些词

- **子查询（subquery）**：写在另一条查询内部的 SELECT。外层查询把子查询的结果当作数据用。子查询可以当作一张临时表（放在 FROM 里），也可以当作真值测试（放在 WHERE 里）。
- **反连接（anti-join）**："找出左表中在右表**找不到**搭档的行"。它是 LEFT JOIN 之后筛 NULL 行的逻辑等价物，但用 `NOT EXISTS` 表达更直接：对每一行问"存在匹配吗？不存在就留下"。
- **EXISTS / NOT EXISTS**：只关心子查询**能不能查出至少一行**，查出来说 TRUE，查不出说 FALSE。子查询里 `SELECT 1` 的 `1` 只是占位符——EXISTS 根本不看选的是什么列，只看行数是不是零。它对 NULL 免疫（NULL 行匹配不上就是匹配不上，不会产生"半真半假"）。
- **相关子查询（correlated subquery）**：子查询里引用了外层的列（比如 `o.customerId = c.id` 里的 `c.id`）。它不是算一次完事，而是**外层每来一行，就按这一行的值重新算一遍**子查询，像内层循环。
- **NOT IN 的 NULL 陷阱**：`x NOT IN (1, NULL)` 对 x=2 来说：`2=1` 是 FALSE，`2=NULL` 是 UNKNOWN，合起来 UNKNOWN，再取反还是 UNKNOWN——WHERE 只留 TRUE，于是 x=2 被**踢出结果**。只要子查询结果里混进一个 NULL，NOT IN 就会把本该保留的行全部冤杀。NOT EXISTS 没有这个问题，所以表达"不存在"优先用它。
- **CTE（Common Table Expression，`WITH x AS (...)`）**：给一段子查询起个名字，后面当普通表用。作用纯粹是**让人读得懂、让中间结果可单独验证**——它不是性能魔法，“正确性与复杂度”部分 特意强调"别指望 CTE 语法本身更快"。
- **双键连接**：连接条件写两个等号（`ON m.departmentId=e.departmentId AND e.salary=m.highest`），两个条件同时成立才算配上。部门最高薪题靠它保留所有并列者。
- **双重 NOT EXISTS（关系除法）**：`不存在(产品 p 满足：不存在(客户 c 买 p 的记录))` 翻译成人话是"没有任何一个产品是他没买过的"，也就是"每个产品他都买过"。这种"全体/所有"语义在 SQL 里标准答案就是双重否定，叫关系除法。
- **DISTINCT 与 UNION/UNION ALL（本章知识点清单里的两兄弟）**：DISTINCT 把结果中重复的行捏成一行；UNION 拼两个查询结果并自动去重，UNION ALL 拼接但保留重复。拿购买表演示语义差别最直观：`SELECT customer_id FROM Customer WHERE product_key=1 UNION SELECT customer_id FROM Customer WHERE product_key=2` 给的是"买过 1 或买过 2 的客户号去重版"（这里会是 1,2,3），把 UNION 换成 UNION ALL 则保留每一次出现（行数回到购买记录数）。本章三段代码实际用的是 DISTINCT（买齐产品题里，客户重复买同一产品不能算多条"合格记录"），UNION 适合"两个来源合一张名单"的场景。
- **空真（vacuous truth）**："所有产品都买过"在产品表为空时自动成立——一个产品都没有，当然谈不上缺哪个。“自动测试”部分 最后一条断言专门验证这个数学边界。

## 3. 思路推导：从小例子开始

- **Step 1（从不订购）**：需求是"没有任何一条订单指向他"。对每个客户跑一遍相关子查询"存在 customerId=他id 的订单吗"，NOT EXISTS 取反。子查询命中 `(2, NULL)` 那行也没关系：NULL 谁也配不上，不影响任何人。
- **Step 2（NOT IN 为什么不行）**：直觉写法 `id NOT IN (SELECT customerId FROM Orders)`，而子查询结果里躺着 NULL。`NOT IN` 的算法是"拿外层的值和列表里的**每一个**元素比，全部比出 FALSE 才算 NOT IN 成立；任何一个比出 UNKNOWN，整个判断就悬成 UNKNOWN"。逐个算：

| 外层 id | 和 1 比 | 和 NULL 比 | 整体 IN 结果 | 取反（NOT IN） | WHERE 放行吗 | 应该吗 |
|---|---|---|---|---|---|---|
| 1 | TRUE | UNKNOWN | TRUE | FALSE | 不放 | 不该放（他下过单），碰巧对 |
| 2 | FALSE | UNKNOWN | UNKNOWN | UNKNOWN | 不放 | **冤枉**，他没下过单该保留 |
| 3 | FALSE | UNKNOWN | UNKNOWN | UNKNOWN | 不放 | **冤枉** |

结果是空集——“运行示例”部分 的对比表展示的正是这个：NOT EXISTS 给 `[B, C]`，含 NULL 的 NOT IN 给 `[]`。一个 NULL 毒倒全部候选行，而且不报错、不警告，静悄悄返回空结果。
- **Step 3（部门最高薪）**：分两步想，先算"每个部门的最高工资是多少"（GROUP BY 部门取 MAX，这是 CTE `maximum`），再回到员工表，用**部门 + 工资双键**把"工资恰好等于本部门最高值"的人全部连出来。并列的人因为双键都能配上，一个不丢。最后再连 Department 表拿部门名。
- **Step 4（买齐所有产品）**：把"买齐所有"翻译成"不存在没买过的产品"：外层 NOT EXISTS 说"找不到这样的产品 p"，内层 NOT EXISTS 说"找不到他买 p 的记录"。内层找不到=他没买过 p；外层找不到=没有他没买过的 p=他全买了。重复购买天然无害：内层只要找到一条购买记录就说不成立，买三遍和买一遍效果相同。

**手算演示 1（反连接，Orders 三行：客户1、客户NULL、客户1）**：

| 客户 | 子查询"存在他的订单吗" | NOT EXISTS 结果 |
|---|---|---|
| A(1) | TRUE（订单 1、3 都指向他） | FALSE，出局 |
| B(2) | FALSE（没有任何订单的 customerId=2；那行 NULL 也配不上） | TRUE，保留 |
| C(3) | FALSE（同上） | TRUE，保留 |

**手算演示 2（部门最高薪）**：第一步 GROUP BY 算出 `maximum = {(部门1, 100), (部门2, 80)}`。第二步双键连回 Employee：Ada(部门1,100) 配上 (1,100) ✓；Ben(部门1,100) ✓（并列者同样配上）；Cal(部门1,90) 配不上（90≠100）✗；Dee(部门2,80) ✓。输出三行，R&D 两人并列都在。

**手算演示 3（双重否定，产品 {1,2}）**：

| 客户 | 考察产品 1：存在购买记录？ | 考察产品 2：存在购买记录？ | 存在"没买过的产品"吗 | 结论 |
|---|---|---|---|---|
| 1 | 有（买了两次也无妨） | 有 | 不存在 | 买齐了，保留 |
| 2 | 有 | **没有** | 存在（产品 2） | 出局 |
| 3 | **没有** | 有 | 存在（产品 1） | 出局 |

## 4. 核心代码：query_customers_without_orders.sql

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```sql
SELECT c.name AS Customers
FROM Customers AS c
WHERE NOT EXISTS (SELECT 1 FROM Orders AS o WHERE o.customerId = c.id);
```

### 逐段读懂代码

“分段实现”部分：三个查询 + 建库脚本，套路与前两章相同（SQL 文本按 `.sql` 文件名存在字典里，`query_rows` 在真 SQLite 上执行）。

**查询 1：从不订购的客户（183）**

```sql
SELECT c.name AS Customers
FROM Customers AS c
WHERE NOT EXISTS (SELECT 1 FROM Orders AS o WHERE o.customerId = c.id);
```

外层逐个客户检查；括号里的相关子查询引用了外层的 `c.id`，含义是"这个客户存在订单吗"。`SELECT 1` 是占位——EXISTS 只看行数不看内容。整句 WHERE 读作"不存在他的任何订单"。那行 customerId 为 NULL 的订单在这里完全无害：NULL=c.id 永远配不上，它既不属于 A 也不属于 B，更不会把任何比较结果污染成 UNKNOWN 再扩散（这正是 NOT EXISTS 对 NOT IN 的本质优势：判断只依赖"有没有行匹配"，每个匹配判断是独立的两值结论，不存在 UNKNOWN 的连锁反应）。

**查询 2：部门最高薪（184）**

```sql
WITH maximum AS (
    SELECT departmentId, MAX(salary) AS highest
    FROM Employee GROUP BY departmentId
)
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee AS e
JOIN maximum AS m ON m.departmentId = e.departmentId AND e.salary = m.highest
JOIN Department AS d ON d.id = e.departmentId;
```

`WITH maximum AS (...)` 就是 CTE：先把"每个部门的最高工资"算成一张两列表（部门号、最高值），起名 `maximum`，主查询里当普通表 JOIN 两次。第一次 JOIN 的条件是**双键**（部门相等**且**工资等于该部门最高值）——这一步决定并列者全保留；第二次 JOIN 拿部门名。把复杂查询拆成"先算小表、再连接"两段，每段都能单独跑、单独验证，这就是本章"可验证中间关系"的示范。

**查询 3：买齐所有产品（1045）**

```sql
SELECT DISTINCT c.customer_id
FROM Customer AS c
WHERE NOT EXISTS (
    SELECT 1 FROM Product AS p
    WHERE NOT EXISTS (
        SELECT 1 FROM Customer AS bought
        WHERE bought.customer_id = c.customer_id AND bought.product_key = p.product_key
    )
);
```

从里往外读：最内层"存在 `bought`（某个购买记录）同时满足客户号是他、产品号是 p"——即"他买过 p"。中层 `WHERE NOT EXISTS(最内层)` 圈出"他没买过的产品"。外层 `NOT EXISTS(中层)` 要求"这样的产品一个都不存在"。三层套完 = "他每种产品都买过"。`DISTINCT` 收尾：购买表是多重集（他可能买同一产品多次，比如客户 1 的 `(1,1),(1,1)`），但合格名单里每个客户只需要出现一次。

**fixture 数据的雷区设计**：Orders 里那行 `(2, NULL)` 专门喂给 NOT IN 陷阱；Employee 里 Ada/Ben 并列 100 专门测并列保留；Customer 购买表里客户 1 和客户 2 的重复购买专门验证 DISTINCT 和"看有没有、不看几次"。

## 5. 分段实现：按顺序运行

**本章接口：** `query_customers_without_orders.sql`、`query_department_max.sql`、`query_bought_all.sql`

### 导入本章使用的库

In [1]:
import sqlite3

### SQL_QUERIES

In [2]:
SQL_QUERIES = {
    'query_customers_without_orders.sql': r'''SELECT c.name AS Customers
FROM Customers AS c
WHERE NOT EXISTS (SELECT 1 FROM Orders AS o WHERE o.customerId = c.id);''',
    'query_department_max.sql': r'''WITH maximum AS (
    SELECT departmentId, MAX(salary) AS highest
    FROM Employee GROUP BY departmentId
)
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee AS e
JOIN maximum AS m ON m.departmentId = e.departmentId AND e.salary = m.highest
JOIN Department AS d ON d.id = e.departmentId;''',
    'query_bought_all.sql': r'''SELECT DISTINCT c.customer_id
FROM Customer AS c
WHERE NOT EXISTS (
    SELECT 1 FROM Product AS p
    WHERE NOT EXISTS (
        SELECT 1 FROM Customer AS bought
        WHERE bought.customer_id = c.customer_id AND bought.product_key = p.product_key
    )
);''',
}

### create_sqlite_fixture

In [3]:
def create_sqlite_fixture():
    connection = sqlite3.connect(':memory:')
    connection.executescript("\nCREATE TABLE Customers(id INTEGER PRIMARY KEY,name TEXT);\nCREATE TABLE Orders(id INTEGER PRIMARY KEY,customerId INTEGER);\nINSERT INTO Customers VALUES (1,'A'),(2,'B'),(3,'C');\nINSERT INTO Orders VALUES (1,1),(2,NULL),(3,1);\nCREATE TABLE Department(id INTEGER PRIMARY KEY,name TEXT);\nCREATE TABLE Employee(id INTEGER PRIMARY KEY,name TEXT,salary INTEGER,departmentId INTEGER);\nINSERT INTO Department VALUES (1,'R&D'),(2,'Sales');\nINSERT INTO Employee VALUES (1,'Ada',100,1),(2,'Ben',100,1),(3,'Cal',90,1),(4,'Dee',80,2);\nCREATE TABLE Product(product_key INTEGER PRIMARY KEY);\nCREATE TABLE Customer(customer_id INTEGER,product_key INTEGER);\nINSERT INTO Product VALUES (1),(2);\nINSERT INTO Customer VALUES (1,1),(1,1),(1,2),(2,1),(2,1),(3,2);\n")
    return connection

### query_rows

In [4]:
def query_rows(connection, name):
    return connection.execute(SQL_QUERIES[name]).fetchall()

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

db=create_sqlite_fixture()
show_table(['反连接方式','实际结果'],[('NOT EXISTS',query_rows(db,'query_customers_without_orders.sql')),('含NULL的NOT IN',db.execute('SELECT name FROM Customers WHERE id NOT IN (SELECT customerId FROM Orders)').fetchall())])
for name in ['query_department_max.sql','query_bought_all.sql']:
    cursor=db.execute(SQL_QUERIES[name]); print(name); show_table([x[0] for x in cursor.description],cursor.fetchall())
db.close()

反连接方式,实际结果
NOT EXISTS,"[('B',), ('C',)]"
含NULL的NOT IN,[]


query_department_max.sql


Department,Employee,Salary
R&D,Ada,100
R&D,Ben,100
Sales,Dee,80


query_bought_all.sql


customer_id
1


## 7. 正确性、适用条件与复杂度

NOT EXISTS只测试是否存在至少一条匹配，不受无关NULL和重复订单影响。聚合最高薪再按部门与工资双键连接，恰保留该部门所有达到最大值的行。双重否定等价于对每个产品都存在购买记录；产品表为空时条件真，返回购买表中可识别的所有客户ID，不凭空创造客户全集。

**代价：** 相关子查询在无索引时可能重复扫描，数据库可优化为连接。教学重点是逻辑语义正确；需要规模优化时为关联键建立索引并查看计划，而不是以CTE语法本身声称更快。

### 展开理解

**NOT EXISTS 为什么稳？** 它的判断只有两个出口："至少一行匹配"（TRUE）或"零行匹配"（FALSE），没有第三态。行里有没有 NULL、有没有重复，都不影响"存在与否"这个事实——NULL 行就是一条不匹配的行而已。而 NOT IN 是拿外层值和子查询结果**逐个比较**，任何一个比较吐出 UNKNOWN 都会污染整行的最终判断，所以一个 NULL 就能废掉整个查询。

**双键连接为什么恰好保留所有并列者？** `maximum` 表里每个部门恰有一行"最高值"；员工行的工资若等于本部门最高值就能配上（不管有几个人同分都能各自配上一次），不等于就配不上。于是"配上的集合"恰好就是"该部门所有拿最高薪的人"，不多不少。对比错误做法"每部门随机取一行"（比如某些数据库的 LIMIT 1 写法）会随机丢掉并列者——题目要求的是全部。

**双重否定为什么等于"全都买过"？** 一层一层翻："存在他没买过的产品"的反面是"他没买过的产品不存在"，也就是"每个产品他都买过"。这个翻译在产品表为空时也成立：没有产品，"没买过的产品"自然一个都不存在，条件为真——这就是空真，“正确性与复杂度”部分 特意说明此时返回的是"购买表里能认出来的所有客户"，而不是凭空发明一个客户全集（外层 FROM 是 Customer 购买表，所以只发购买记录里出现过的人）。

再演示一遍"重复购买为什么天然无害"：客户 1 的购买记录是 `(1,1),(1,1),(1,2)`。考产品 1 时，最内层子查询找到第一条 `(1,1)` 就立刻返回 TRUE（EXISTS 找到一行就收工，绝不数第二遍），第二条 `(1,1)` 对结果毫无影响。所以"买过没有"是布尔判断，不是计数——这也是为什么最终还要套一层 DISTINCT：同一个合格的客户在外层循环里只会出现一次，但保险起见把"重复"这个话题在输出关口也堵死。

**为什么 CTE 版部门最高薪比"一条大查询"更容易验证？** `maximum` 这张中间表可以单独拷出来跑：`SELECT * FROM maximum` 立刻看到 `{(1,100),(2,80)}`，肉眼核对每部门的最高值；主查询出问题时代错也能定位到"是最高值算错了还是连接键写错了"。把查询拆成有名有姓的段落，就是在给未来的自己留调试入口——这正是本章目标"拆成可验证中间关系"的含义。

**复杂度是多少？** 相关子查询在没索引时，外层每行都要把内层表扫一遍，最坏是"外层行数 × 内层行数"的重复扫描，比如 1 万客户配 10 万订单就是 10 亿次探测。但 EXISTS 有个好朋友叫索引：给 `Orders.customerId` 建索引后，"存在吗"变成一次索引查找，整个查询降为外层行数 × O(log 内层行数)。数据库的优化器也可能自动把相关子查询改写成连接计划，所以**看执行计划（EXPLAIN QUERY PLAN）才是判断代价的正道**；CTE 只是让你把逻辑写清楚，它本身既不加快也不减慢——“正确性与复杂度”部分 特意拆穿了"CTE 更快"的迷信。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 每条断言逐个拆（验证方式：Python assert + 真 SQLite 执行比对）：

- `set(query_rows(db,'query_customers_without_orders.sql'))=={('B',),('C',)}`：**反连接正常值 + NULL 免疫**。A 下过单必须消失，B/C 必须保留；同时 Orders 里那行 NULL 订单不得干扰任何人的判定（若误用 NOT IN，这里会得到空集，当场翻车）。
- `set(query_rows(db,'query_department_max.sql'))=={('R&D','Ada',100),('R&D','Ben',100),('Sales','Dee',80)}`：**并列保留**。R&D 两个 100 必须双双入选（专抓"每组随便取一行"的错误写法），Cal(90) 必须落选，Sales 只有一人也要正常出线。用 set 比对，因为没写 ORDER BY。
- `query_rows(db,'query_bought_all.sql')==[(1,)]`：**关系除法正常值**。客户 1 买齐（含重复购买，重复不得产生重复输出行，这也在验 DISTINCT）；客户 2、3 各缺一件必须落选。这里用列表直接 `==[(1,)]` 而不是 set——结果恰只有一行，顺序无从谈起，两种写法等价，但当你想顺便断言"只有一个答案"时列表写法更直白。
- `DELETE FROM Orders` 后 `len(...)==3`：**边界变动**。订单全删，三个客户全都"从不订购"，输出必须从 2 人变 3 人——验证查询真的在逐行动态判断，而不是背答案。
- `DELETE FROM Product` 后 `set(...)=={(1,),(2,),(3,)}`：**空真边界**。产品表清空后"买齐所有产品"的条件对每个人都自动成立，输出必须是购买表里出现过的全部三个客户。这条同时验证了两件事：空集下的数学语义（条件为真）和"不凭空造客户"（外层 FROM 限定范围）。

In [6]:
db = create_sqlite_fixture()

assert set(query_rows(db, 'query_customers_without_orders.sql')) == {('B',), ('C',)}

assert set(query_rows(db, 'query_department_max.sql')) == {('R&D', 'Ada', 100), ('R&D', 'Ben', 100), ('Sales', 'Dee', 80)}

assert query_rows(db, 'query_bought_all.sql') == [(1,)]

db.execute('DELETE FROM Orders')

assert len(query_rows(db, 'query_customers_without_orders.sql')) == 3

db.execute('DELETE FROM Product')

assert set(query_rows(db, 'query_bought_all.sql')) == {(1,), (2,), (3,)}

db.close()

print('N154: 所有本章断言通过')

N154: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 用NOT EXISTS避免NOT IN与NULL陷阱。

**练习 2：** 比较最高薪并列处理。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（用 NOT EXISTS 避免 NOT IN 与 NULL 陷阱）**：提示——现成的讲稿素材就是 “运行示例”部分 的对比表和第三节的逐值手算：对 `id NOT IN (1, NULL)` 把 id=1/2/3 三个值各算一遍 TRUE/FALSE/UNKNOWN，看着"本该保留的行全灭"。验证方法：在 fixture 上分别跑 `NOT EXISTS` 版和 `NOT IN` 版对照输出（“运行示例”部分 已经跑给你看了：`[B,C]` 对 `[]`）。再想边界：子查询结果一个 NULL 都没有时两种写法结果一致，陷阱只在混入 NULL 时引爆——所以工程上与其赌"这列永远没有 NULL"，不如养成直接写 NOT EXISTS 的习惯。
- **练习 2（比较最高薪并列处理）**：提示——拿 Ada/Ben 并列 100 这组数据当实验台。正确基准是双键连接版（输出 3 行）。对比几种常见错误：`GROUP BY departmentId` 后随便挑一个人（SQLite 挑谁不确定，可能丢 Ben）；用窗口函数 `ROW_NUMBER()`（按工资降序编号取第 1 名——会把并列的 Ben 编成第 2 名丢掉）；用 `RANK()`（并列同名次，能保留，值得想通为什么）。手算 Ada/Ben/Cal 三人在 ROW_NUMBER 和 RANK 两种编号下的序号，就能看清谁丢并列、谁不丢。边界：全部门工资全相同（全都是最高薪，全要保留）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

本章包含真实的 SQL／Shell／JavaScript／线程程序，Python 部分负责创建示例、调用运行时和断言。嵌入的程序源码也在下面完整显示；缺少相应运行时应安装依赖，不能用模拟结果替代。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
import sqlite3

SQL_QUERIES = {
    'query_customers_without_orders.sql': r'''SELECT c.name AS Customers
FROM Customers AS c
WHERE NOT EXISTS (SELECT 1 FROM Orders AS o WHERE o.customerId = c.id);''',
    'query_department_max.sql': r'''WITH maximum AS (
    SELECT departmentId, MAX(salary) AS highest
    FROM Employee GROUP BY departmentId
)
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee AS e
JOIN maximum AS m ON m.departmentId = e.departmentId AND e.salary = m.highest
JOIN Department AS d ON d.id = e.departmentId;''',
    'query_bought_all.sql': r'''SELECT DISTINCT c.customer_id
FROM Customer AS c
WHERE NOT EXISTS (
    SELECT 1 FROM Product AS p
    WHERE NOT EXISTS (
        SELECT 1 FROM Customer AS bought
        WHERE bought.customer_id = c.customer_id AND bought.product_key = p.product_key
    )
);''',
}

def create_sqlite_fixture():
    connection = sqlite3.connect(':memory:')
    connection.executescript("\nCREATE TABLE Customers(id INTEGER PRIMARY KEY,name TEXT);\nCREATE TABLE Orders(id INTEGER PRIMARY KEY,customerId INTEGER);\nINSERT INTO Customers VALUES (1,'A'),(2,'B'),(3,'C');\nINSERT INTO Orders VALUES (1,1),(2,NULL),(3,1);\nCREATE TABLE Department(id INTEGER PRIMARY KEY,name TEXT);\nCREATE TABLE Employee(id INTEGER PRIMARY KEY,name TEXT,salary INTEGER,departmentId INTEGER);\nINSERT INTO Department VALUES (1,'R&D'),(2,'Sales');\nINSERT INTO Employee VALUES (1,'Ada',100,1),(2,'Ben',100,1),(3,'Cal',90,1),(4,'Dee',80,2);\nCREATE TABLE Product(product_key INTEGER PRIMARY KEY);\nCREATE TABLE Customer(customer_id INTEGER,product_key INTEGER);\nINSERT INTO Product VALUES (1),(2);\nINSERT INTO Customer VALUES (1,1),(1,1),(1,2),(2,1),(2,1),(3,2);\n")
    return connection

def query_rows(connection, name):
    return connection.execute(SQL_QUERIES[name]).fetchall()

# 示例与回归测试
db = create_sqlite_fixture()

assert set(query_rows(db, 'query_customers_without_orders.sql')) == {('B',), ('C',)}

assert set(query_rows(db, 'query_department_max.sql')) == {('R&D', 'Ada', 100), ('R&D', 'Ben', 100), ('Sales', 'Dee', 80)}

assert query_rows(db, 'query_bought_all.sql') == [(1,)]

db.execute('DELETE FROM Orders')

assert len(query_rows(db, 'query_customers_without_orders.sql')) == 3

db.execute('DELETE FROM Product')

assert set(query_rows(db, 'query_bought_all.sql')) == {(1,), (2,), (3,)}

db.close()

print('N154: 所有本章断言通过')

N154: 所有本章断言通过


## 11. 代表题与迁移

- **183. Customers Who Never Order**：对应查询 1——练反连接的 NOT EXISTS 写法，核心收获是 NOT IN 遇 NULL 全军覆没的陷阱。
- **184. Department Highest Salary**：对应查询 2——练 CTE 拆两步 + 部门与工资双键连接保留全部并列最高薪。
- **1045. Customers Who Bought All Products**：对应查询 3——练双重 NOT EXISTS 表达"对所有产品都存在购买记录"的关系除法，顺带 DISTINCT 处理重复购买与空产品表的空真边界。

三题练完的检验标准：你能不看笔记，把"NOT IN 为什么被 NULL 废掉""并列最高薪怎么全保留""买齐所有怎么翻成双重否定"各讲两分钟，这章就算到手了。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。